# BƯỚC 1 & 2: KHẢO SÁT & KIỂM TRA CHẤT LƯỢNG DỮ LIỆU THÔ (DATA QUALITY & PROFILING)
## DỰ ÁN: IT JOB MARKET INTELLIGENCE
---
**Mục tiêu của Notebook:**
1. **Trích xuất dữ liệu cần thiết (Feature Selection)** từ kho dữ liệu thô Bronze (bronze_listings + bronze_details_valid).
2. **Kiểm tra cấu trúc và kiểu dữ liệu (Schema & Data Types)** của toàn bộ các trường tuyển dụng.
3. **Đo lường giá trị khuyết (Missing / Null Report)**: Thống kê số lượng Null/None và chuỗi rỗng trên từng cột.
4. **Kiểm tra trùng lặp (Duplicates Report)**: Phát hiện trùng lặp theo ID nguồn, URL bài đăng và thực thể công việc [Công ty + Chức danh].
5. Đưa ra cơ sở thực tế để thực hiện bước Tiền xử lý & Làm sạch dữ liệu tiếp theo.

In [1]:
import os
import json
import pandas as pd
import numpy as np
from IPython.display import display

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print(' Môi trường phân tích dữ liệu đã sẵn sàng!')

 Môi trường phân tích dữ liệu đã sẵn sàng!


## 1. Nạp Dữ Liệu Master Từ Tầng Bronze

Đọc 2 tệp tích lũy chuẩn từ `data/bronze/`:
- `bronze_listings_master.json`: Bảng danh mục tin tuyển dụng (kèm tags, logo, trạng thái crawl).
- `bronze_details_valid.json`: Bảng chi tiết việc làm hợp lệ sau khi qua bộ kiểm định Data Quality.

In [2]:
# Duong dan toi 2 file Bronze Master (tuong thich moi thu muc lam viec)
def resolve_path(rel_subpath):
    candidates = [
        rel_subpath,
        f"../{rel_subpath}",
        f"../../{rel_subpath}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return os.path.abspath(rel_subpath)

listings_path = resolve_path('data/bronze/bronze_listings_master.json')
details_path = resolve_path('data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    listings_data = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    details_data = json.load(f)

df_listings = pd.DataFrame(listings_data)
df_details = pd.DataFrame(details_data)

print(f'Dang doc Listings tu: {listings_path}')
print(f'Dang doc Details tu : {details_path}\n')
print(f'Bang Listings: {len(df_listings):,} dong, {df_listings.shape[1]} cot')
print(f'Bang Details : {len(df_details):,} dong, {df_details.shape[1]} cot\n')

# Thong ke so luong tin theo tung san
summary_sources = pd.DataFrame({
    'So Listings': df_listings['source'].value_counts(),
    'So Details Valid': df_details['source'].value_counts()
})
display(summary_sources)


Dang doc Listings tu: c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_listings_master.json
Dang doc Details tu : c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_details_valid.json

Bang Listings: 4,579 dong, 20 cot
Bang Details : 4,534 dong, 35 cot



,So Listings,So Details Valid
source,,
topcv,2873,2828
vietnamworks,1539,1539
topdev,167,167


## 2. Ghép 2 Bảng (Join Listings + Details)

Ghép theo cặp khóa định danh duy nhất: `['source', 'source_job_id']`.
- **Mục tiêu**: Bổ sung các cột giá trị từ Listings mà Details có thể chưa có (như `company_logo_url`, `skills_raw`, `posted_at_raw`, `raw_file_name` của listing).
- Giữ nguyên toàn bộ các trường nội dung chi tiết sâu (`overview_raw`, `responsibilities_raw`, `requirements_raw`, `benefits_raw`, `work_mode_raw`, `company_size_raw`...) từ Details.

In [3]:
# Chọn các cột bổ sung từ Listings (tránh xung đột trùng lặp cột)
listing_extra_cols = ['source', 'source_job_id', 'company_logo_url', 'skills_raw', 'posted_at_raw', 'raw_file_name']
df_list_sub = df_listings[listing_extra_cols].drop_duplicates(subset=['source', 'source_job_id']).rename(
    columns={'raw_file_name': 'listing_file_name'}
)

# Ghép bảng Details với Listings theo ['source', 'source_job_id']
df_merged = pd.merge(
    df_details,
    df_list_sub,
    on=['source', 'source_job_id'],
    how='inner'
)

print(f' Kết quả ghép bảng (Join):')
print(f'  • Tổng số dòng sau khi ghép : {len(df_merged):,} dòng')
print(f'  • Tổng số cột sau khi ghép  : {df_merged.shape[1]} cột')
print(f'  • Tỷ lệ khớp khóa (Match)  : {len(df_merged)/len(df_details)*100:.2f}% (Khớp hoàn toàn 100%)')

 Kết quả ghép bảng (Join):
  • Tổng số dòng sau khi ghép : 4,534 dòng
  • Tổng số cột sau khi ghép  : 39 cột
  • Tỷ lệ khớp khóa (Match)  : 100.00% (Khớp hoàn toàn 100%)


## 3. Thống Kê Chi Tiết: Info, Missing Values (Null / Rỗng), và Duplicates (Dup)

Khảo sát toàn diện chất lượng dữ liệu của bảng sau khi ghép:
1. **Cấu trúc & Kiểu dữ liệu (`.info()`)**.
2. **Thống kê giá trị khuyết (`null` và chuỗi rỗng `""`)**: Tính số lượng và tỷ lệ % khuyết trên từng trường dữ liệu.
3. **Kiểm tra trùng lặp (`duplicates`)**: Kiểm tra trùng theo ID nguồn, theo URL, và theo cặp chức danh - công ty.

In [4]:
# 1. Hiển thị thông tin tổng quan kiểu dữ liệu
print('--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---')
df_merged.info(verbose=True, show_counts=True)

# 2. Thống kê Missing Values (Null / None và Chuỗi Rỗng "")
null_counts = df_merged.isnull().sum()
empty_counts = (df_merged == '').sum()
total_missing = null_counts + empty_counts
missing_pct = (total_missing / len(df_merged) * 100).round(2)

df_missing_report = pd.DataFrame({
    'Kiểu Dữ Liệu': df_merged.dtypes.astype(str),
    'Số lượng Null/None': null_counts,
    'Số lượng Chuỗi Rỗng ("")': empty_counts,
    'Tổng Số Khuyết': total_missing,
    'Tỷ Lệ Khuyết (%)': missing_pct
}).sort_values(by='Tỷ Lệ Khuyết (%)', ascending=False)

print('\n--- 2. BẢNG THỐNG KÊ GIÁ TRỊ KHUYẾT (MISSING / NULL REPORT) ---')
display(df_missing_report)

# 3. Kiểm tra Trùng lặp (Duplicates Report)
dup_source_id = df_merged.duplicated(subset=['source', 'source_job_id']).sum()
dup_job_url = df_merged.duplicated(subset=['job_url']).sum()
dup_title_company = df_merged.duplicated(subset=['title_raw', 'company_name_raw']).sum()

print('--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---')
df_dup_report = pd.DataFrame({
    'Tiêu Chí Trùng Lặp': [
        'Theo cặp khóa nguồn [source, source_job_id]',
        'Theo đường dẫn [job_url]',
        'Theo cùng vị trí & cùng công ty [title_raw, company_name_raw]'
    ],
    'Số Bản Ghi Trùng': [dup_source_id, dup_job_url, dup_title_company],
    'Tỷ Lệ Trùng (%)': [
        round(dup_source_id / len(df_merged) * 100, 2),
        round(dup_job_url / len(df_merged) * 100, 2),
        round(dup_title_company / len(df_merged) * 100, 2)
    ]
})
display(df_dup_report)

--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 4534 entries, 0 to 4533
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   detail_id             4534 non-null   str   
 1   source                4534 non-null   str   
 2   source_job_id         4534 non-null   str   
 3   job_url               4534 non-null   str   
 4   title_raw             4534 non-null   str   
 5   company_name_raw      4534 non-null   str   
 6   company_url           4534 non-null   str   
 7   company_size_raw      4534 non-null   str   
 8   company_location_raw  4534 non-null   str   
 9   industry_raw          4534 non-null   str   
 10  salary_raw            4534 non-null   str   
 11  experience_raw        4534 non-null   str   
 12  education_raw         4534 non-null   str   
 13  location_raw          4534 non-null   str   
 14  work_mode_raw         4534 non-null   str   
 15  employmen

,Kiểu Dữ Liệu,Số lượng Null/None,"Số lượng Chuỗi Rỗng ("""")",Tổng Số Khuyết,Tỷ Lệ Khuyết (%)
preferred_skills_raw,str,0,4534,4534,100.00
updated_at_raw,str,0,2995,2995,66.06
overview_raw,str,0,2676,2676,59.02
company_size_raw,str,0,967,967,21.33
education_raw,str,0,746,746,16.45
working_time_raw,str,0,279,279,6.15
company_logo_url,str,0,35,35,0.77
work_mode_raw,str,0,35,35,0.77
requirements_raw,str,0,20,20,0.44
required_skills_raw,str,0,5,5,0.11


--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---


,Tiêu Chí Trùng Lặp,Số Bản Ghi Trùng,Tỷ Lệ Trùng (%)
0,"Theo cặp khóa nguồn [source, source_job_id]",0,0.00
1,Theo đường dẫn [job_url],0,0.00
2,"Theo cùng vị trí & cùng công ty [title_raw, company_name...",189,4.17


## 4. Xem Mẫu Dữ Liệu Sau Khi Ghép (Toàn Bộ 39 Cột - Dễ Quan Sát)

Vì dữ liệu sau khi ghép có **39 cột** (bao gồm cả các trường text dài và metadata), nếu để một bảng phẳng nằm ngang sẽ rất khó quan sát. Chúng ta tổ chức hiển thị theo các cách sau để bạn dễ kiểm tra nhất:

1.  **Cách 1: Hiển thị Bảng Chuyển Vị (Transposed `.T`)**: Chuyển toàn bộ 39 cột thành danh sách dọc. Mỗi tin là một cột  Dễ dàng cuộn dọc đọc trọn vẹn từng trường mà không bị tràn màn hình.
2.  **Cách 2: Hiển thị 10 Dòng Theo Từng Nhóm Nghiệp Vụ**: Phân tách 39 cột thành 5 nhóm có cấu trúc (*Định danh, Công ty, Lương & Đãi ngộ, Kỹ năng, Nội dung JD, Data Quality*).
3.  **Cách 3: Bảng Phẳng Đầy Đủ (Toàn bộ 39 cột x 10 dòng)**: Mở rộng `display.max_columns = None` để bạn có thể cuộn ngang xem tất cả các cột nếu cần.

In [5]:
# --- CÁCH 1: HIỂN THỊ DẠNG CHUYỂN VỊ (TRANSPOSED VIEW) ---
# 39 cột sẽ hiển thị theo chiều dọc (Index), các mẫu tin hiển thị theo cột
print(' [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:')
sample_3 = df_merged.sample(3, random_state=42)
display(sample_3.T)


 [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:


,544,1356,1569
detail_id,topcv_2325071,vietnamworks_2089428,topcv_2071877
source,topcv,vietnamworks,topcv
source_job_id,2325071,2089428,2071877
job_url,https://www.topcv.vn/viec-lam/nhan-vien-thiet-ke-2d-3d/2...,https://www.vietnamworks.com/r-d-senior-engineer-embedde...,https://www.topcv.vn/viec-lam/application-engineer-maste...
title_raw,Nhân Viên Thiết Kế 2D/3D,R&D Senior Engineer (Embedded Software),Application Engineer - Mastercam Software
company_name_raw,CÔNG TY TNHH VÀNG BẠC ĐÁ QUÝ HOÀ HIẾU,"Panasonic Appliances Vietnam Co.,Ltd.",CÔNG TY TNHH SOLID & SOFT
company_url,https://www.topcv.vn/cong-ty/hoa-hieu-jewelry-ahoa/19277...,https://www.vietnamworks.com/nha-tuyen-dung/panasonic-ap...,https://www.topcv.vn/cong-ty/cong-ty-tnhh-solid-soft/433...
company_size_raw,100-499 nhân viên,1.000-4.999,25-99 nhân viên
company_location_raw,"151-155 Trần Tuấn Khải, Phường 05, Quận 5, Thành phố Hồ ...","Thang Long Industrial Park, Dong Anh, Ha Noi | Thang Lon...","Lầu 7, Tòa nhà Parkson Paragon, 3 Nguyễn Lương Bằng, Phư..."
industry_raw,"Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...",Điện/Điện tử (Công Nghệ Thông Tin/Viễn Thông > Phần Mềm ...,"Sản xuất, Cơ khí / Chế tạo / Tự động hóa, Công nghệ cao"


In [ ]:
# ---  XEM MẪU 10 DÒNG THEO TỪNG NHÓM NGHIỆP VỤ ---
sample_10 = df_merged.sample(10, random_state=42)

# Nhóm 1: Định danh & Truy vết nguồn (7 cột)
group_identity = ['source', 'source_job_id', 'job_url', 'crawled_at', 'raw_date', 'raw_file_name', 'listing_file_name']
print(' NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)')
display(sample_10[group_identity])

# Nhóm 2: Thông tin Công ty (7 cột)
group_company = ['company_name_raw', 'company_url', 'company_logo_url', 'company_size_raw', 'company_location_raw', 'industry_raw', 'location_raw']
print('\n NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)')
display(sample_10[group_company])

# Nhóm 3: Vị trí, Lương & Đãi ngộ tuyển dụng (8 cột)
group_job_specs = ['title_raw', 'salary_raw', 'experience_raw', 'education_raw', 'work_mode_raw', 'employment_type_raw', 'working_time_raw', 'deadline_raw']
print('\n NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)')
display(sample_10[group_job_specs])

# Nhóm 4: Kỹ năng & Ngày tháng (6 cột)
group_skills = ['skills_raw', 'required_skills_raw', 'preferred_skills_raw', 'posted_at_raw', 'published_at_raw', 'updated_at_raw']
print('\n NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)')
display(sample_10[group_skills])

# Nhóm 5: Nội dung chi tiết văn bản (4 cột)
group_texts = ['overview_raw', 'responsibilities_raw', 'requirements_raw', 'benefits_raw']
print('\n NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)')
display(sample_10[['title_raw'] + group_texts].map(lambda x: (str(x)[:80] + '...') if len(str(x)) > 80 else str(x)))


 NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)


,source,source_job_id,job_url,crawled_at,raw_date,raw_file_name,listing_file_name
544,topcv,2325071,https://www.topcv.vn/viec-lam/nhan-vien-thiet-ke-2d-3d/2...,2026-10-11T20:41:14.432281,2026-10-11,TCV009.json,page_008.json
1356,vietnamworks,2089428,https://www.vietnamworks.com/r-d-senior-engineer-embedde...,2026-10-11T20:37:01.635116,2026-10-11,VNW007.json,page_007.json
1569,topcv,2071877,https://www.topcv.vn/viec-lam/application-engineer-maste...,2026-10-12T10:05:33.619676,2026-10-12,TCV001.json,page_001.json
23,topcv,2303376,https://www.topcv.vn/brand/vnresource1209/tuyen-dung/lap...,2026-10-11T00:10:45.878955,2026-10-11,TCV001.json,page_001.json
2230,vietnamworks,2115735,https://www.vietnamworks.com/nhan-vien-thiet-ke-ui-ux-we...,2026-10-12T21:04:10.654806,2026-10-12,VNW004.json,page_004.json
3646,vietnamworks,2103754,https://www.vietnamworks.com/technical-support-engineer-...,2026-10-24T17:11:36.205524,2026-10-24,VNW001.json,page_001.json
4511,vietnamworks,2124851,https://www.vietnamworks.com/head-of-delivery-1793259889...,2026-10-29T21:32:55.146793,2026-10-29,VNW002.json,page_002.json
2669,topcv,2330748,https://www.topcv.vn/viec-lam/fullstack-developer/233074...,2026-10-14T22:13:19.266527,2026-10-14,TCV002.json,page_002.json
3664,topcv,2298677,https://www.topcv.vn/viec-lam/product-business-analyst/2...,2026-10-25T12:38:50.067999,2026-10-25,TCV001.json,page_001.json
4496,topdev,2140360,https://topdev.vn/viec-lam/network-expansion-associate-j...,2026-10-29T21:32:37.854757,2026-10-29,TC002.json,page_002.json



 NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)


,company_name_raw,company_url,company_logo_url,company_size_raw,company_location_raw,industry_raw,location_raw
544,CÔNG TY TNHH VÀNG BẠC ĐÁ QUÝ HOÀ HIẾU,https://www.topcv.vn/cong-ty/hoa-hieu-jewelry-ahoa/19277...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,100-499 nhân viên,"151-155 Trần Tuấn Khải, Phường 05, Quận 5, Thành phố Hồ ...","Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...","151-155 Trần Tuấn Khải, Phường An Đông, Hồ Chí Minh"
1356,"Panasonic Appliances Vietnam Co.,Ltd.",https://www.vietnamworks.com/nha-tuyen-dung/panasonic-ap...,https://images.vietnamworks.com/pictureofcompany/db/1092...,1.000-4.999,"Thang Long Industrial Park, Dong Anh, Ha Noi | Thang Lon...",Điện/Điện tử (Công Nghệ Thông Tin/Viễn Thông > Phần Mềm ...,"Lô G2, KCN Thăng Long II, Nguyễn Văn Linh, Hưng Yên"
1569,CÔNG TY TNHH SOLID & SOFT,https://www.topcv.vn/cong-ty/cong-ty-tnhh-solid-soft/433...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,25-99 nhân viên,"Lầu 7, Tòa nhà Parkson Paragon, 3 Nguyễn Lương Bằng, Phư...","Sản xuất, Cơ khí / Chế tạo / Tự động hóa, Công nghệ cao","Epic Tower, 19 Duy Tan, Cau Giay Ward, Phường Cầu Giấy, ..."
23,CÔNG TY CỔ PHẦN TÀI NGUYÊN TRI THỨC VIỆT NĂNG,https://www.topcv.vn/brand/vnresource1209?id=134822,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,,"6 Nguyễn Trung Trực, Phường Bình Lợi Trung, Hồ Chí Minh","IT phần mềm, Công nghệ thông tin","6 Nguyễn Trung Trực, Phường Bình Lợi Trung, Hồ Chí Minh"
2230,Công Ty TNHH WEEDSOFT VIETNAM,https://www.vietnamworks.com/nha-tuyen-dung/cong-ty-tnhh...,https://images.vietnamworks.com/pictureofcompany/e5/1145...,Ít hơn 10,145 Điện Biên Phủ Phường Tân Định TPHCM,Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,145 Điện Biên Phủ Phường Tân Định
3646,HYUNDAI HT VINA,https://www.vietnamworks.com/nha-tuyen-dung/hyundai-ht-v...,https://images.vietnamworks.com/pictureofcompany/5d/1138...,Ít hơn 10,"PTS Saigon Office B/D, 118 Huỳnh Tấn Phát, Tân Thuận Tây...",Điện/Điện tử (Khoa Học & Kỹ Thuật > Kỹ Thuật Điện/Điện Tử),"HYUNDAI HT VINA HANOI OFFICE, PHẠM HÙNG STREET, SÔNG ĐÀ ..."
4511,Xenia Tech,https://www.vietnamworks.com/nha-tuyen-dung/xenia-tech-c...,,,Ho Chi Minh; Da Nang,Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,Ho Chi Minh; Da Nang
2669,CÔNG TY TNHH NGÔI SAO DŨNG CẢM,https://www.topcv.vn/cong-ty/cong-ty-tnhh-ngoi-sao-dung-...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,100-499 nhân viên,"37 Lê Văn Lương, Thanh Xuân, Hà Nội, Việt Nam","IT phần mềm, Công nghệ thông tin","37 Lê Văn Lương, Phường Thanh Xuân, Hà Nội"
3664,Công ty cổ phần Công nghệ Kỹ thuật Vtech,https://www.topcv.vn/cong-ty/cong-ty-co-phan-cong-nghe-k...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,25-99 nhân viên,Tầng 10 - Tòa nhà Lilama 10 - 56 Tố Hữu - Trung Văn - Na...,"IT phần mềm, Công nghệ thông tin","Tầng 11 - Tòa nhà Lilama 10 - 56 Tố Hữu, Phường Đại Mỗ, ..."
4496,CÔNG TY TNHH DIALECTICA VIETNAM,https://topdev.vn/nha-tuyen-dung/cong-ty-tnhh-dialectica...,https://assets.saramin.vn/images/2026/10/23/Saramin-1792...,Hơn 1000 Nhân viên,"VRG Building, 177 Hai Ba Trung, Xuan Hoa Ward, ThÃ nh ph...","Dịch vụ doanh nghiệp, B2B Services",Hồ Chí Minh



 NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)


,title_raw,salary_raw,experience_raw,education_raw,work_mode_raw,employment_type_raw,working_time_raw,deadline_raw
544,Nhân Viên Thiết Kế 2D/3D,Thoả thuận,2 năm,Cao Đẳng trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-07T23:59:59+07:00
1356,R&D Senior Engineer (Embedded Software),Thương lượng,1 năm,Cử nhân,Onsite,Nhân viên,Toàn thời gian,2026-10-11T23:59:59+07:00
1569,Application Engineer - Mastercam Software,Thoả thuận,2 năm,Đại Học trở lên,Onsite,Nhân viên,Toàn thời gian,2026-10-18T23:59:59+07:00
23,Lập Trình Viên (.NET),Thoả thuận,2 năm,,Onsite,Nhân viên,Toàn thời gian,2026-10-16T23:59:59+07:00
2230,Nhân viên thiết kế UI / UX Website,12tr-18tr ₫/tháng,1 năm,,Onsite,Nhân viên,Toàn thời gian,2026-11-04T23:59:59+07:00
3646,Technical Support Engineer - Hanoi,"$ 600-1,200 /tháng",1 năm,Cử nhân,Onsite,Nhân viên,Toàn thời gian,2026-11-20T23:59:59+07:00
4511,Head Of Delivery,70tr-90tr ₫/tháng,6 năm,,Onsite,Trưởng phòng,Toàn thời gian,2026-11-28T23:59:59+07:00
2669,FullStack Developer,Tới 30 triệu,2 năm,Đại Học trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-13T23:59:59+07:00
3664,Product Business Analyst,12000000 - 15000000,1 năm,Đại Học trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-20T23:59:59+07:00
4496,Network Expansion Associate (Japanese/Korean speaking),Negotiable MONTH,"Không yêu cầu, 3 năm",,,Mới tốt nghiệp,,2026-11-22



 NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)


,skills_raw,required_skills_raw,preferred_skills_raw,posted_at_raw,published_at_raw,updated_at_raw
544,"2 năm kinh nghiệm chuyên môn, Xem tất cả việc làm UI/UX ...","2 năm kinh nghiệm chuyên môn, Xem tất cả việc làm UI/UX ...",,2026-10-08,2026-10-08,
1356,"Debugging Tools Usage, Lập Trình Nhúng, Firmware, C/C++ ...","Debugging Tools Usage, Lập Trình Nhúng, Firmware, C/C++ ...",,2026-09-11T15:10:50+07:00,2026-09-11T15:10:50+07:00,2026-09-11T15:10:50+07:00
1569,"2 năm kinh nghiệm chuyên môn, Xem tất cả việc làm Kỹ sư ...","2 năm kinh nghiệm chuyên môn, Xem tất cả việc làm Kỹ sư ...",,2026-09-18,2026-09-18,
23,"2 năm kinh nghiệm chuyên môn, Software Engineer, IT - Ph...","2 năm kinh nghiệm chuyên môn, Software Engineer, IT - Ph...",,2026-09-16,2026-09-16,
2230,"Thiết Kế UX, User Research, Usability Testing, A/B testi...","Thiết Kế UX, User Research, Usability Testing, A/B testi...",,2026-10-05T16:44:49+07:00,2026-10-05T16:44:49+07:00,2026-10-06T16:08:39+07:00
3646,"English, Network Technology, Team Work, It Technical Sup...","English, Network Technology, Team Work, It Technical Sup...",,2026-10-21T10:30:05+07:00,2026-10-21T10:30:05+07:00,2026-10-21T10:30:05+07:00
4511,"Agile, Performance Management, Technical Lead, Software ...","Agile, Performance Management, Technical Lead, Software ...",,2026-10-29T14:44:49+07:00,2026-10-29T14:44:49+07:00,2026-10-29T14:44:49+07:00
2669,"2 năm kinh nghiệm chuyên môn, Fullstack Developer, IT - ...","2 năm kinh nghiệm chuyên môn, Fullstack Developer, IT - ...",,2026-10-14,2026-10-14,
3664,"1 năm kinh nghiệm chuyên môn, Business Analyst (Phân tíc...","1 năm kinh nghiệm chuyên môn, Business Analyst (Phân tíc...",,2026-10-21,2026-10-21,
4496,"Japanese, Korean, Account Manager","Japanese, Korean, Account Manager",,2026-10-23,2026-10-23,



 NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)


,title_raw,overview_raw,responsibilities_raw,requirements_raw,benefits_raw
544,Nhân Viên Thiết Kế 2D/3D,,Thiết kế mẫu trang sức 2D/3D theo yêu cầu và định hướng ...,"• Tốt nghiệp Cao đẳng/Đại học chuyên ngành Mỹ thuật, Th...","Mức lương cạnh tranh, hấp dẫn, thỏa thuận theo năng lực ..."
1356,R&D Senior Engineer (Embedded Software),"Panasonic Appliances Vietnam Co., Ltd. (PAPVN), 100% inv...",We are looking for potential candidates who are willing ...,- University's degree in Electrical/Electronic Engineeri...,"Thưởng: Gói lương cạnh tranh, được xét tăng lương hằng n..."
1569,Application Engineer - Mastercam Software,,COMPUTER AIDED MANUFACTURING (CAM) SOFTWARE -\nMASTERCAM...,Fluent in English communication.\nGraduated from Univers...,International working environment.\nSalary and bonus com...
23,Lập Trình Viên (.NET),,"Tham gia phát triển, bảo trì và nâng cấp các hệ thống ph...",Tốt nghiệp Cao đẳng/Đại học chuyên ngành Công nghệ Thông...,"Mức lương cạnh tranh dựa trên năng lực và kinh nghiệm, k..."
2230,Nhân viên thiết kế UI / UX Website,,- Thiết kế và tối ưu UX/UI cho sản phẩm FLEXG.Page – nền...,- CV bằng tiếng Anh\n- Portfolio thể hiện các dự án UX/U...,Thưởng: Lương tháng 13\nNghỉ phép có lương: Nghỉ phép\nM...
3646,Technical Support Engineer - Hanoi,HYUNDAI HT VINA LLC is the Vietnam corporation of HYUNDA...,1. Technical Documentation & Support\n• Prepare product ...,"• Bachelor's degree or above\n(Electrical Engineering, E...","Thưởng: Tet Bonus\nChăm sóc sức khoẻ: Social insurance, ..."
4511,Head Of Delivery,,Role Purpose\nThis is a high-impact leadership role cove...,"●​ 6+ years hands-on software development experience, wi...",Khác: <p>TBD</p>
2669,FullStack Developer,,"Thiết kế, xây dựng và triển khai các internal tool và hệ...","Có từ 2 đến 4 năm kinh nghiệm phát triển phần mềm, trong...",Mức lương cạnh tranh lên đến 30M gross.\nThưởng cố định ...
3664,Product Business Analyst,,VTech đang tìm kiếm Product Business Analyst tham gia ph...,Yêu cầu bắt buộc\n- Có từ 01 năm kinh nghiệm Business An...,"Cơ hội phát triển theo lộ trình Product BA, Senior Produ..."
4496,Network Expansion Associate (Japanese/Korean speaking),$3f,Your day-to-day responsibilities will include:\nBrief An...,"WHAT WE LOOK FOR\nExperience: 0-3 years of experience, p...",WHY JOIN US?\nSkill Mastery: Gain transferable skills in...


In [9]:
# ---  BẢNG PHẲNG ĐẦY ĐỦ TOÀN BỘ 39 CỘT (SAMPLE 10 DÒNG) ---
print(' [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):')
display(sample_10)


 [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):


,detail_id,source,source_job_id,job_url,title_raw,company_name_raw,company_url,company_size_raw,company_location_raw,industry_raw,salary_raw,experience_raw,education_raw,location_raw,work_mode_raw,employment_type_raw,working_time_raw,overview_raw,responsibilities_raw,requirements_raw,benefits_raw,required_skills_raw,preferred_skills_raw,published_at_raw,updated_at_raw,deadline_raw,crawled_at,raw_file_name,raw_date,dq_status,dq_errors,dq_checked_at,_dq_status,_dq_errors,_dq_checked_at,company_logo_url,skills_raw,posted_at_raw,listing_file_name
544,topcv_2325071,topcv,2325071,https://www.topcv.vn/viec-lam/nhan-vien-thiet-ke-2d-3d/2...,Nhân Viên Thiết Kế 2D/3D,CÔNG TY TNHH VÀNG BẠC ĐÁ QUÝ HOÀ HIẾU,https://www.topcv.vn/cong-ty/hoa-hieu-jewelry-ahoa/19277...,100-499 nhân viên,"151-155 Trần Tuấn Khải, Phường 05, Quận 5, Thành phố Hồ ...","Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...",Thoả thuận,2 năm,Cao Đẳng trở lên,"151-155 Trần Tuấn Khải, Phường An Đông, Hồ Chí Minh",Onsite,Nhân viên,Toàn thời gian,,Thiết kế mẫu trang sức 2D/3D theo yêu cầu và định hướng ...,"• Tốt nghiệp Cao đẳng/Đại học chuyên ngành Mỹ thuật, Th...","Mức lương cạnh tranh, hấp dẫn, thỏa thuận theo năng lực ...","2 năm kinh nghiệm chuyên môn, Xem tất cả việc làm UI/UX ...",,2026-10-08,,2026-11-07T23:59:59+07:00,2026-10-11T20:41:14.432281,TCV009.json,2026-10-11,valid,[],2026-10-29T22:02:49.769437,valid,[],2026-10-29T22:02:49.769437,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,"2 năm kinh nghiệm chuyên môn, Xem tất cả việc làm UI/UX ...",2026-10-08,page_008.json
1356,vietnamworks_2089428,vietnamworks,2089428,https://www.vietnamworks.com/r-d-senior-engineer-embedde...,R&D Senior Engineer (Embedded Software),"Panasonic Appliances Vietnam Co.,Ltd.",https://www.vietnamworks.com/nha-tuyen-dung/panasonic-ap...,1.000-4.999,"Thang Long Industrial Park, Dong Anh, Ha Noi | Thang Lon...",Điện/Điện tử (Công Nghệ Thông Tin/Viễn Thông > Phần Mềm ...,Thương lượng,1 năm,Cử nhân,"Lô G2, KCN Thăng Long II, Nguyễn Văn Linh, Hưng Yên",Onsite,Nhân viên,Toàn thời gian,"Panasonic Appliances Vietnam Co., Ltd. (PAPVN), 100% inv...",We are looking for potential candidates who are willing ...,- University's degree in Electrical/Electronic Engineeri...,"Thưởng: Gói lương cạnh tranh, được xét tăng lương hằng n...","Debugging Tools Usage, Lập Trình Nhúng, Firmware, C/C++ ...",,2026-09-11T15:10:50+07:00,2026-09-11T15:10:50+07:00,2026-10-11T23:59:59+07:00,2026-10-11T20:37:01.635116,VNW007.json,2026-10-11,valid,[],2026-10-29T22:02:51.641903,valid,[],2026-10-29T22:02:51.641903,https://images.vietnamworks.com/pictureofcompany/db/1092...,"Debugging Tools Usage, Lập Trình Nhúng, Firmware, C/C++ ...",2026-09-11T15:10:50+07:00,page_007.json
1569,topcv_2071877,topcv,2071877,https://www.topcv.vn/viec-lam/application-engineer-maste...,Application Engineer - Mastercam Software,CÔNG TY TNHH SOLID & SOFT,https://www.topcv.vn/cong-ty/cong-ty-tnhh-solid-soft/433...,25-99 nhân viên,"Lầu 7, Tòa nhà Parkson Paragon, 3 Nguyễn Lương Bằng, Phư...","Sản xuất, Cơ khí / Chế tạo / Tự động hóa, Công nghệ cao",Thoả thuận,2 năm,Đại Học trở lên,"Epic Tower, 19 Duy Tan, Cau Giay Ward, Phường Cầu Giấy, ...",Onsite,Nhân viên,Toàn thời gian,,COMPUTER AIDED MANUFACTURING (CAM) SOFTWARE -\nMASTERCAM...,Fluent in English communication.\nGraduated from Univers...,International working environment.\nSalary and bonus com...,"2 năm kinh nghiệm chuyên môn, Xem tất cả việc làm Kỹ sư ...",,2026-09-18,,2026-10-18T23:59:59+07:00,2026-10-12T10:05:33.619676,TCV001.json,2026-10-12,valid,[],2026-10-29T22:02:49.775900,valid,[],2026-10-29T22:02:49.775900,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,"2 năm kinh nghiệm chuyên môn, Xem tất cả việc làm Kỹ sư ...",2026-09-18,page_001.json
23,topcv_2303376,topcv,2303376,https://www.topcv.vn/brand/vnresource1209/tuyen-dung/lap...,Lập Trình Viên (.NET),CÔNG TY CỔ PHẦN TÀI NGUYÊN TRI THỨC VIỆT NĂNG,https://www.topcv.vn/brand/vnresource1209?id=134822,,"6 Nguyễn 